# ScatterRad crop `.npz` visualizer

This notebook helps inspect preprocessed crop files in `crops/*.npz`.

Each crop file contains **both** arrays:
- `image` (float32)
- `mask` (uint8)
- `meta` (dict-like payload)

So we do not need separate files for image and mask. Keeping them in one archive keeps them aligned and simplifies I/O.


In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (15, 5)


In [ ]:
# Configure dataset location
dataset_name = 'Dataset001_SpineCTMets'
pre_root = os.environ.get('SCATTERRAD_PREPROCESSED', '/home/matthias.walle/data/scratch/matthias/data/ScatterRad/scatterrad_preprocessed')
crops_dir = Path(pre_root) / dataset_name / 'crops'

crop_files = sorted(crops_dir.glob('*.npz'))
print(f'crops_dir: {crops_dir}')
print(f'num crop files: {len(crop_files)}')
print('first 10 files:')
for p in crop_files[:10]:
    print(' -', p.name)


In [ ]:
# Pick a file by index
idx = 0
crop_path = crop_files[idx]

with np.load(crop_path, allow_pickle=True) as payload:
    image = payload['image'].astype(np.float32)
    mask = payload['mask'].astype(np.uint8)
    meta = payload['meta'].item()

print('file:', crop_path.name)
print('image shape:', image.shape, image.dtype, f'range=({image.min():.2f}, {image.max():.2f})')
print('mask shape:', mask.shape, mask.dtype, 'mask voxels:', int(mask.sum()))
print('meta:', meta)


In [ ]:
def show_slice(z=None, alpha=0.35):
    if z is None:
        z = image.shape[0] // 2

    base = image[z]
    m = mask[z] > 0

    fig, ax = plt.subplots(1, 3)

    ax[0].imshow(base, cmap='gray')
    ax[0].set_title(f'Image z={z}')
    ax[0].axis('off')

    ax[1].imshow(m, cmap='Reds')
    ax[1].set_title('Mask')
    ax[1].axis('off')

    ax[2].imshow(base, cmap='gray')
    ax[2].imshow(m.astype(float), cmap='autumn', alpha=alpha)
    ax[2].set_title('Overlay')
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

show_slice()


In [ ]:
# Optional: interactive slider (if ipywidgets is available)
try:
    import ipywidgets as widgets
    widgets.interact(show_slice, z=widgets.IntSlider(min=0, max=image.shape[0]-1, step=1, value=image.shape[0]//2));
except Exception as e:
    print('ipywidgets not available:', e)
    print('Use show_slice(z=<index>) manually.')


## Scatter cache visualizer

Each scatter cache file `scatter_cache/*.npy` contains precomputed scattering coefficients with shape `(C, D, H, W)` where C is the number of scattering channels.

In [ ]:
scatter_dir = Path(pre_root) / dataset_name / 'scatter_cache'
scatter_files = sorted(scatter_dir.glob('*.npy'))
print(f'scatter_dir: {scatter_dir}')
print(f'num scatter files: {len(scatter_files)}')
print('first 10 files:')
for p in scatter_files[:10]:
    print(' -', p.name)

In [ ]:
# Load a scatter file matching the currently loaded crop (or pick by index)
scatter_path = scatter_dir / (crop_path.stem + '.npy')
if not scatter_path.exists():
    scatter_path = scatter_files[idx]

scatter = np.load(scatter_path)  # shape: (C, D, H, W)
print('file:', scatter_path.name)
print('scatter shape (C, D, H, W):', scatter.shape)
print(f'range: ({scatter.min():.4f}, {scatter.max():.4f})  mean: {scatter.mean():.4f}')

In [ ]:
def show_scatter_slice(z=None, log_scale=True):
    """Show all scatter channels at a given depth slice."""
    n_channels = scatter.shape[0]
    if z is None:
        z = scatter.shape[1] // 2
    z = min(z, scatter.shape[1] - 1)

    ncols = min(n_channels, 8)
    nrows = (n_channels + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).reshape(nrows, ncols)

    for c in range(n_channels):
        ax = axes[c // ncols, c % ncols]
        sl = scatter[c, z]
        if log_scale:
            sl = np.log1p(np.abs(sl)) * np.sign(sl)
        ax.imshow(sl, cmap='RdBu_r', interpolation='nearest')
        ax.set_title(f'ch {c}', fontsize=8)
        ax.axis('off')

    # Hide unused axes
    for c in range(n_channels, nrows * ncols):
        axes[c // ncols, c % ncols].axis('off')

    scale_label = 'log1p' if log_scale else 'linear'
    fig.suptitle(f'{scatter_path.name}  z={z}  ({scale_label} scale)', fontsize=10)
    plt.tight_layout()
    plt.show()

show_scatter_slice()

In [ ]:
# Interactive slider (if ipywidgets available)
try:
    import ipywidgets as widgets
    widgets.interact(
        show_scatter_slice,
        z=widgets.IntSlider(min=0, max=scatter.shape[1] - 1, step=1, value=scatter.shape[1] // 2),
        log_scale=widgets.Checkbox(value=True, description='log scale'),
    )
except Exception as e:
    print('ipywidgets not available:', e)
    print('Use show_scatter_slice(z=<index>, log_scale=True) manually.')